In [1]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

service = Service(GeckoDriverManager().install())
driver = webdriver.Firefox(service=service)

driver.get("https://pokedle.com/classic")

wait = WebDriverWait(driver, 10)

# accepter cookies UNE FOIS
try:
    cookie_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Consent')]"))
    )
    cookie_btn.click()
except:
    pass

time.sleep(10)

def get_input():
    return wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='text']"))
    )

input_box = get_input()

def search_pokemon(name):
    global input_box
    try:
        input_box.clear()
    except:
        input_box = get_input()  # re-find si stale

    input_box.send_keys(name)
    input_box.send_keys(Keys.ENTER)


def get_cases():

    rows = driver.find_elements(By.CSS_SELECTOR, ".flex.w-fit")
    results = []

    for row in rows:
        cards = row.find_elements(By.CSS_SELECTOR, ".result-card")
        values = []

        for card in cards:
            imgs = card.find_elements(By.TAG_NAME, "img")
            if imgs:
                titles = [img.get_attribute("title") for img in imgs if img.get_attribute("title") if img.get_attribute("title").strip()!="pokeballcardback"]
                alts = [
                        img.get_attribute("alt")
                        for img in imgs
                        if img.get_attribute("alt")
                        and "poke ball" not in img.get_attribute("alt").lower()
                        and "arrow" not in img.get_attribute("alt").lower()
                    ]
                if titles:
                    values.append(titles[0].strip())
                    continue

                if alts:
                    values.append(",".join([a.replace(" ", "") for a in alts]))
                    continue

            try:
                text = card.find_element(By.TAG_NAME, "p").text.strip()
                if text:
                    values.append(text)
                    continue
            except:
                pass

            text = card.text.strip()
            if text:
                values.append(text)

        if values:
            results.append(values)

    return results

pokemon_list = []

with open("pokemon.txt", "r", encoding="utf-8") as f:
    pokemon_list = [line.strip() for line in f if line.strip()]

with open("resultats.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    writer.writerow([
        "Pokemon",
        "Type 1",
        "Type 2",
        "Stade d'évolution",
        "Entièrement évolué",
        "Couleur",
        "Habitat",
        "Gen"
    ])
    counter=0
    for name in pokemon_list:
        if counter==10:
            driver.refresh()
            counter=0
        else:
            counter+=1
        try:
            search_pokemon(name)
            time.sleep(3)
            data = get_cases()
            data[0][0] = name

            if len(data) > 1:
                row = data[0]

                # nettoyage
                row = [v.replace("\n", "").strip() for v in row]

                writer.writerow(row)

            print(f"{name} OK")

        except Exception as e:
            print(f"Erreur avec {name} :", e)

# fermer
driver.quit()

Mew OK
Germignon OK
Macronium OK
Méganium OK
